In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable


# From different folder Importing defined schemas

In [0]:
%run /Workspace/Users/kharitejareddy997@gmail.com/setup/utlities

# Identifing the paths for specific files

In [0]:
%fs ls s3://orders-ecommerce-de/raw

path,name,size,modificationTime
s3://orders-ecommerce-de/raw/customers.csv,customers.csv,245761,1782049384000
s3://orders-ecommerce-de/raw/products.csv,products.csv,83070,1782049384000
s3://orders-ecommerce-de/raw/sales_reps.csv,sales_reps.csv,20062,1782049384000
s3://orders-ecommerce-de/raw/transactions.csv,transactions.csv,625057,1782049384000


In [0]:
# reading the data from s3 Bucket 
df = spark.read.csv('s3://orders-ecommerce-de/raw/products.csv',header = True, inferSchema=True)

In [0]:
# Writing the data to the bronze schema
df.write.mode('overwrite').format('delta').saveAsTable('ecommerce.bronze.products')

In [0]:
df_silver = spark.read.table('ecommerce.bronze.products')

# Performing Data Transformations

In [0]:
df_silver = df_silver.withColumn('Category',
                     f.when(f.col("category") == 'home & garder', 'Home & Garden')
                     .when(f.col("category") == 'Clothng', 'Clothing')
                     .when(f.col("category") == 'Electroniks', 'Electronics')
                     .when(f.col("category") == 'sports', 'Sports')
                     .when(f.col("category") == 'toys', 'Toys')
                     )

In [0]:
df_silver = df_silver.withColumn('Category',
                                 f.when((f.col("subcategory") == 'Men')| (f.col("subcategory") == 'Women') | (f.col("subcategory") == 'Kids') | (f.col("subcategory") == 'Shoes'), 'Clothing')
                                .when((f.col("subcategory") == 'Audio') | (f.col("subcategory") == 'Smartphones') | (f.col("subcategory") == 'Accessories') | (f.col("subcategory") == 'Laptops'),'Electronics')
                                .when((f.col("subcategory") == 'Water Sports') | (f.col("subcategory") == 'Fitness') | (f.col("subcategory") == 'Outdoor') | (f.col("subcategory") == 'Team Sports'), 'Sports')
                                .when((f.col("subcategory") == 'Educational') | (f.col("subcategory") == 'Board Games') | (f.col("subcategory") == 'Action Figures') | (f.col("subcategory") == 'Puzzles'), 'Toys')
                                .when((f.col("subcategory") == 'Decor') | (f.col("subcategory") == 'Kitchen') | (f.col("subcategory") == 'Tools') | (f.col("subcategory") == 'Furniture'), 'Home & Garden')
                                
                                )

In [0]:
# Converting Negative values to Positive
df_silver = df_silver.withColumn('unit_price', f.round(f.abs(f.col('unit_price')), 2))

In [0]:
# Converting Negative values to Positive and rounding to 2 digits
df_silver = df_silver.withColumn('cost_price', f.round(f.abs(f.col("cost_price")), 2))

In [0]:
# Filling The Null Values in supplier_id column with 'Unknown'
df_silver = df_silver.fillna('Unknown', subset=["supplier_id"])

In [0]:
# Converting Negative values to Positive
df_silver = df_silver.withColumn('stock_quantity', f.abs(f.col("stock_quantity")))

# Writing to the silver table

In [0]:
df_silver.write.mode('overwrite').option('mergeSchema', True).format('delta').saveAsTable('ecommerce.silver.products')

In [0]:
display(df_silver)

product_id,product_name,Category,subcategory,unit_price,cost_price,supplier_id,stock_quantity,product_launch_date
PROD_0001,Accessories Elite 622,Electronics,Accessories,442.98,328.58,SUP_39,728,2023-06-08
PROD_0002,Accessories Max 894,Electronics,Accessories,345.67,188.7,SUP_50,631,2023-01-04
PROD_0003,Smartphones Standard 459,Electronics,Smartphones,682.33,284.73,SUP_40,950,2022-02-26
PROD_0004,Water Sports Basic 938,Sports,Water Sports,461.22,245.72,Unknown,905,2023-05-27
PROD_0005,Accessories Basic 934,Electronics,Accessories,887.94,443.65,SUP_26,174,2023-08-04
PROD_0006,Puzzles Max 341,Toys,Puzzles,378.74,269.83,SUP_50,22,2022-06-20
PROD_0007,Decor Pro 249,Home & Garden,Decor,284.18,226.11,SUP_21,724,2023-01-12
PROD_0008,Decor Standard 302,Home & Garden,Decor,279.83,208.2,SUP_2,916,2022-06-11
PROD_0009,Laptops Max 716,Electronics,Laptops,574.61,290.49,SUP_41,234,2022-09-19
PROD_0010,Fitness Max 878,Sports,Fitness,534.63,403.95,SUP_18,12,2022-04-16


In [0]:
dim_products = spark.sql("""
                         select DISTINCT product_id, product_name, category, subcategory, supplier_id, product_launch_date
                         from ecommerce.silver.products
                         """)

In [0]:
dim_products.write.mode('overwrite').format('delta').saveAsTable('ecommerce.gold.dim_products')